In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import os

os.makedirs("../output", exist_ok=True)

panel = pd.read_csv("../data/stacked_event_panel.csv")
panel = panel[(panel["k"] >= -6) & (panel["k"] <= 12)].copy()
panel["post"] = (panel["k"] >= 0).astype(int)
panel["k_cat"] = panel["k"].astype(str)
panel.loc[panel["k"] == -1, "k_cat"] = "ref"

no_co = panel[panel["state"] != "CO"].copy()

print("=" * 100)
print(f"LEAVE-COLORADO-OUT ROBUSTNESS (N={len(no_co)}, states remaining: {sorted(no_co['state'].unique())})")
print("=" * 100)
print(f"Full panel: {len(panel)} obs, {panel['state'].nunique()} states, CO alone = "
      f"{(panel['state']=='CO').sum()} obs ({(panel['state']=='CO').mean()*100:.0f}% of panel)")
print(f"Events remaining with AZ/CT only: {sorted(no_co['event_id'].unique())}")
print(f"(Barkley and Shaq were CO-only events -- they drop out entirely with CO removed.)\n")

for label, formula in {
    "(1) Basic, no CO": "log_handle ~ post + C(state_event)",
    "(2) + trend, no CO": "log_handle ~ post + C(state_event) + C(state_event):t_idx",
}.items():
    if "t_idx" in formula:
        no_co["t_idx"] = no_co.groupby("state_event")["month"].rank(method="dense")
    m_cluster = smf.ols(formula, data=no_co).fit(cov_type="cluster", cov_kwds={"groups": no_co["state"]})
    m_hc1 = smf.ols(formula, data=no_co).fit(cov_type="HC1")
    print(f"{label}")
    print(f"  post coefficient : {m_cluster.params['post']:+.3f}")
    print(f"  SE (cluster/state, 2 clusters): {m_cluster.bse['post']:.3f}   p={m_cluster.pvalues['post']:.3f}")
    print(f"  SE (HC1, non-clustered)       : {m_hc1.bse['post']:.3f}   p={m_hc1.pvalues['post']:.3f}")
    print(f"  implied handle change: {(np.exp(m_cluster.params['post']) - 1) * 100:.1f}%")
    print(f"  R-squared: {m_cluster.rsquared:.3f}   df_resid: {int(m_cluster.df_resid)}\n")

model = smf.ols(
    "log_handle ~ C(k_cat, Treatment(reference='ref')) + C(state_event)",
    data=no_co,
).fit(cov_type="cluster", cov_kwds={"groups": no_co["state"]})

ks = sorted(no_co.loc[no_co["k"] != -1, "k"].unique())
rows = []
for k in ks:
    term = f"C(k_cat, Treatment(reference='ref'))[T.{k}]"
    if term in model.params.index:
        rows.append({"k": k, "beta_k": model.params[term], "se": model.bse[term], "p": model.pvalues[term]})
coef_table = pd.DataFrame(rows)
coef_table["sig"] = coef_table["p"].apply(lambda p: "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.10 else "")
print("Event-study path, no CO:")
print(coef_table.to_string(index=False, formatters={"beta_k": "{:.3f}".format, "se": "{:.3f}".format, "p": "{:.3f}".format}))
print(f"\nR-squared: {model.rsquared:.3f}   N obs: {int(model.nobs)}   df_resid: {int(model.df_resid)}")

coef_table.to_csv("../output/leave_co_out_event_study_coefficients.csv", index=False)
with open("../output/leave_co_out_summary.txt", "w") as f:
    f.write(str(model.summary()))
print("\nSaved: ../output/leave_co_out_event_study_coefficients.csv, ../output/leave_co_out_summary.txt")

LEAVE-COLORADO-OUT ROBUSTNESS (N=110, states remaining: ['AZ', 'CT'])
Full panel: 224 obs, 3 states, CO alone = 114 obs (51% of panel)
Events remaining with AZ/CT only: ['Jamie Foxx x BetMGM (2023-09)', 'Kevin Hart x DraftKings (2023-08)', 'Peyton & Eli Manning x Caesars Sportsbook (2021-11)', 'Rob Gronkowski x FanDuel (2023-01)']
(Barkley and Shaq were CO-only events -- they drop out entirely with CO removed.)

(1) Basic, no CO
  post coefficient : +0.226
  SE (cluster/state, 2 clusters): 0.010   p=0.000
  SE (HC1, non-clustered)       : 0.058   p=0.000
  implied handle change: 25.4%
  R-squared: 0.860   df_resid: 103

(2) + trend, no CO
  post coefficient : +0.228
  SE (cluster/state, 2 clusters): 0.086   p=0.008
  SE (HC1, non-clustered)       : 0.094   p=0.015
  implied handle change: 25.6%
  R-squared: 0.863   df_resid: 97

Event-study path, no CO:
 k beta_k    se     p sig
-6 -0.007 0.134 0.959    
-5  0.047 0.101 0.643    
-4  0.107 0.040 0.007 ***
-3  0.089 0.013 0.000 ***
-2 -

/opt/anaconda3/lib/python3.13/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 23, but rank is 1
  warnings.warn('covariance of constraints does not have full '
